In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR  = '/Users/jackzipper/QSS20/final_project/final_project_data/'
IDP_PATH  = DATA_DIR + 'idp_dat_eastern_drc.csv'
IATI_PATH = DATA_DIR + 'iati-drc-cleaned.csv'
OUT_PATH  = DATA_DIR + 'aid_displacement_merged.csv'

EASTERN_PROVINCES = ['Nord-kivu', 'Sud-kivu', 'Ituri']
NAME_MAP = {'Nord-Kivu': 'Nord-kivu', 'Sud-Kivu': 'Sud-kivu', 'Ituri': 'Ituri'}


# ── Helper functions ───────────────────────────────────────────────────────

def compute_monthly_spend(df):
    """Add a monthly_spend column: total spend divided by project length in months."""
    df = df.copy()
    df['day_length'] = (df['day_end'] - df['day_start']).dt.days
    df['monthly_spend'] = np.where(
        df['day_length'] > 0,
        df['spend'] / (df['day_length'] / 30.44),
        0
    )
    return df


def build_aid_panel(idp, iati_east):
    """Vectorised rolling-window aid construction.

    For each (province, snapshot_month) in the IDP panel, computes:
      - new_project_monthly_spend / new_project_count:
          projects whose start date falls in the prior 6-month window
      - total_active_monthly_spend / total_active_projects:
          projects active (started <= month, ended >= month)
    """
    # Build a slim key table with the 6-month lookback window start
    keys = idp[['admin1_label', 'snapshot_month']].copy()
    keys['window_start'] = keys['snapshot_month'] - pd.DateOffset(months=6)

    # Cross-join keys × projects on province, then filter by date conditions
    crossed = keys.merge(
        iati_east[['location_name', 'day_start', 'day_end', 'monthly_spend']],
        left_on='admin1_label', right_on='location_name',
        how='left'
    )

    grp_cols = ['admin1_label', 'snapshot_month']

    # New projects: started within the prior 6-month window
    new_mask = (
        (crossed['day_start'] >= crossed['window_start']) &
        (crossed['day_start'] <  crossed['snapshot_month'])
    )
    new_agg = (
        crossed[new_mask]
        .groupby(grp_cols)['monthly_spend']
        .agg(new_project_monthly_spend='sum', new_project_count='count')
        .reset_index()
    )

    # Active projects: started on or before month, still ongoing
    active_mask = (
        (crossed['day_start'] <= crossed['snapshot_month']) &
        (crossed['day_end']   >= crossed['snapshot_month'])
    )
    active_agg = (
        crossed[active_mask]
        .groupby(grp_cols)['monthly_spend']
        .agg(total_active_monthly_spend='sum', total_active_projects='count')
        .reset_index()
    )

    # Merge both aggregations back onto the key table
    panel = (
        keys[grp_cols]
        .merge(new_agg,    on=grp_cols, how='left')
        .merge(active_agg, on=grp_cols, how='left')
        .fillna(0)
    )
    return panel

In [2]:
# ── Load data ──────────────────────────────────────────────────────────────
idp  = pd.read_csv(IDP_PATH)
iati = pd.read_csv(IATI_PATH)

print(f'IDP rows loaded:  {len(idp):,}')
print(f'IATI rows loaded: {len(iati):,}')

# Parse dates
idp['snapshot_month'] = pd.to_datetime(idp['snapshot_month'])
iati['day_start'] = pd.to_datetime(iati['day_start'], errors='coerce')
iati['day_end']   = pd.to_datetime(iati['day_end'],   errors='coerce')

# Standardize province names and filter to eastern DRC
iati['location_name'] = iati['location_name'].str.strip().replace(NAME_MAP)
iati_east = iati[iati['location_name'].isin(EASTERN_PROVINCES)].copy()
iati_east = compute_monthly_spend(iati_east)

print(f'IATI eastern projects after filter: {len(iati_east):,}')

IDP rows loaded:  57
IATI rows loaded: 4,917
IATI eastern projects after filter: 511


In [3]:
# ── Build aid panel (vectorised) ───────────────────────────────────────────
df_aid_panel = build_aid_panel(idp, iati_east)
print(f'Aid panel rows: {len(df_aid_panel):,}')

# ── Merge onto IDP panel ───────────────────────────────────────────────────
print(f'\nPre-merge  — IDP rows: {len(idp):,}')
df_merged = pd.merge(idp, df_aid_panel, on=['admin1_label', 'snapshot_month'], how='left')
print(f'Post-merge — merged rows: {len(df_merged):,}')
assert len(df_merged) == len(idp), 'Row count changed after merge — check for duplicate keys'

# ── Log transforms ─────────────────────────────────────────────────────────
df_merged['log_new_project_spend']  = np.log1p(df_merged['new_project_monthly_spend'])
df_merged['log_total_active_spend'] = np.log1p(df_merged['total_active_monthly_spend'])

print(f'\nMissing values in key columns:')
print(df_merged[['log_new_project_spend', 'log_total_active_spend', 'net_monthly_flow']].isnull().sum())
df_merged.head()

Aid panel rows: 57

Pre-merge  — IDP rows: 57
Post-merge — merged rows: 57

Missing values in key columns:
log_new_project_spend     0
log_total_active_spend    0
net_monthly_flow          0
dtype: int64


,admin1_label,snapshot_month,total_displaced,num_sites_displaced,total_returnees,num_sites_returnees,net_monthly_flow,new_project_monthly_spend,new_project_count,total_active_monthly_spend,total_active_projects,log_new_project_spend,log_total_active_spend
0,Ituri,2021-09-01,98078.0,606.0,49189.0,311.0,48889.0,1.932914e+05,3.0,1.317326e+07,32,12.171960,16.393700
1,Ituri,2022-03-01,1951.0,11.0,0.0,0.0,1951.0,1.719128e+06,20.0,1.388762e+07,37,14.357328,16.446508
2,Ituri,2022-04-01,35136.0,2.0,0.0,0.0,35136.0,1.176701e+06,13.0,1.421567e+07,39,13.978226,16.469856
3,Ituri,2022-08-01,11362.0,1.0,42465.0,1.0,-31103.0,4.003435e+06,8.0,1.712940e+07,38,15.202664,16.656307
4,Ituri,2022-12-01,0.0,0.0,13200.0,1.0,-13200.0,3.830606e+06,13.0,1.610884e+07,30,15.158534,16.594879


In [4]:
# ── Save ───────────────────────────────────────────────────────────────────
df_merged.to_csv(OUT_PATH, index=False)
print(f'Saved {len(df_merged):,} rows → {OUT_PATH}')

Saved 57 rows → /Users/jackzipper/QSS20/final_project/final_project_data/aid_displacement_merged.csv
